# RDF and Structure Factor Walkthrough

This notebook is a guided tour through the repository.

It is designed to connect four ideas:

1. how to read or generate atomistic trajectories,
2. how to compute the radial distribution function `g(r)`,
3. how to interpret first peaks, first minima, and coordination numbers,
4. how to transform `g(r)` into the static structure factor `S(k)`.


## 1. Physics Summary

The central real-space structural quantity is the radial distribution function:

$$
g(r) = \frac{\text{actual pair density}}{\text{ideal gas pair density}}
$$

For an isotropic system, the number of neighbors in a spherical shell is

$$
dN = \rho g(r) 4\pi r^2 dr
$$

and the cumulative coordination number is

$$
N(r) = 4\pi\rho \int_0^r g(r') r'^2 dr'.
$$

The reciprocal-space partner is the static structure factor:

$$
S(k) = 1 + 4\pi\rho \int_0^{\infty} [g(r)-1] \frac{\sin(kr)}{kr} r^2 dr.
$$

In this repository, both `g(r)` and `S(k)` are computed by our own code.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

## 2. Import the Project Modules

The notebook reuses the same modules as the example scripts, so it stays aligned with the codebase.


In [ ]:
import pandas as pd
from IPython.display import Image, display

from io_utils import read_xyz
from trajectory import load_frames, run_rdf_analysis, run_multiple_pair_rdfs
from structure_factor import compute_structure_factor_from_rdf, structure_factor_dataframe
from lj_simulation import simulate_lj_fluid


## 3. Argon-like Example: Compute RDF

We start from the bundled Argon-like trajectory and compute `Ar-Ar` RDF. This is the simplest path through the project and mirrors `examples/run_rdf_example.py`.


In [ ]:
argon_frames = load_frames(PROJECT_ROOT / 'data' / 'example_argon.xyz', file_format='xyz')
argon_rdf, argon_analysis = run_rdf_analysis(
    frames=argon_frames,
    atom_type_a='Ar',
    atom_type_b='Ar',
    r_max=5.0,
    bin_width=0.05,
    box_length=10.52,
    use_pbc=True,
    smoothing_window=5,
)

print('First peak r =', argon_analysis.first_peak_r)
print('First minimum r =', argon_analysis.first_minimum_r)
print('First-shell coordination number =', argon_analysis.coordination_number)
pd.DataFrame({'r': argon_rdf.r, 'g_r': argon_rdf.g_r}).head()

If you have already run the example script, the saved figure can be displayed directly.


In [ ]:
argon_figure = PROJECT_ROOT / 'results' / 'rdf_Ar_Ar.png'
if argon_figure.exists():
    display(Image(filename=str(argon_figure)))
else:
    print('Run examples/run_rdf_example.py first to generate the saved figure.')

## 4. From RDF to S(k)

Now we feed that RDF into the structure-factor module. The result is the reciprocal-space view of the same structural information.


In [ ]:
argon_sk = compute_structure_factor_from_rdf(
    rdf_result=argon_rdf,
    k_min=0.1,
    k_max=20.0,
    k_step=0.1,
)

structure_factor_dataframe(argon_sk).head()

In [ ]:
argon_sk_figure = PROJECT_ROOT / 'results' / 'structure_factor_Ar_Ar.png'
if argon_sk_figure.exists():
    display(Image(filename=str(argon_sk_figure)))
else:
    print('Run examples/run_structure_factor_example.py first to generate the saved figure.')

## 5. Water Example: Species-Resolved RDF

For multi-component systems, different pair types carry different structural information. In water, the most common pair RDFs are:

- `O-O`
- `O-H`
- `H-H`


In [ ]:
water_frames = load_frames(PROJECT_ROOT / 'data' / 'example_water.xyz', file_format='xyz')
water_pairs = [
    ('O', 'O', 6.0, 0.05),
    ('O', 'H', 4.0, 0.025),
    ('H', 'H', 4.0, 0.025),
]
water_rdf_batch = run_multiple_pair_rdfs(
    frames=water_frames,
    pair_settings=water_pairs,
    box_length=12.0,
    use_pbc=True,
    smoothing_window=5,
)

summary_rows = []
for pair_key, (rdf_result, analysis) in water_rdf_batch.items():
    summary_rows.append({
        'pair': rdf_result.pair_label,
        'first_peak_r': analysis.first_peak_r,
        'first_minimum_r': analysis.first_minimum_r,
        'coordination_number': analysis.coordination_number,
    })

pd.DataFrame(summary_rows)

## 6. Lennard-Jones Fluid: Generate Your Own Trajectory

The repository can also generate a simple fluid trajectory by itself. This is useful because it closes the loop:

simulation -> trajectory -> RDF -> structure factor.


In [ ]:
lj_simulation = simulate_lj_fluid(
    n_atoms=32,
    reduced_density=0.75,
    reduced_temperature=1.2,
    n_equilibration_sweeps=40,
    n_production_sweeps=80,
    max_displacement=0.18,
    sample_interval=8,
    random_seed=11,
)

print('Sampled frames =', len(lj_simulation.frames))
print('Acceptance ratio =', lj_simulation.acceptance_ratio)
print('Box length =', lj_simulation.box_length)

## 7. Suggested Exercises

Try modifying the notebook or the example scripts to answer the following:

1. How does the RDF change if the water `O-H` bin width becomes smaller?
2. How does the Lennard-Jones acceptance ratio change if `max_displacement` becomes too large?
3. What part of `g(r)` most strongly influences the first peak of `S(k)`?
4. How does increasing the RDF cutoff change the stability of the numerical `S(k)` integral?
